In [1]:
import pandas as pd
df=pd.read_csv("EVSE-B-PowerCombined_Cleaned.csv")

In [2]:
df

,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Attack-Group,Label,interface
0,2023-12-25 22:35:00,978,5.165,1027,5300,idle,syn-flood,dos,attack,ocpp
1,2023-12-25 22:35:00,872,5.161,1009,4980,idle,syn-flood,dos,attack,ocpp
2,2023-12-25 22:35:00,1017,5.165,1029,5300,idle,syn-flood,dos,attack,ocpp
3,2023-12-25 22:35:00,930,5.161,1005,5180,idle,syn-flood,dos,attack,ocpp
4,2023-12-25 22:35:00,958,5.165,1034,5180,idle,syn-flood,dos,attack,ocpp
...,...,...,...,...,...,...,...,...,...,...
114193,2023-12-30 11:51:00,699,5.177,764,2680,idle,backdoor,host-attack,attack,any
114194,2023-12-30 11:51:00,484,5.201,487,2500,idle,backdoor,host-attack,attack,any
114195,2023-12-30 11:51:00,477,5.197,567,2860,idle,backdoor,host-attack,attack,any
114196,2023-12-30 11:51:00,510,5.197,509,2660,idle,backdoor,host-attack,attack,any


In [4]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

# Load the dataset
data = pd.read_csv('EVSE-B-PowerCombined_Cleaned.csv')  # Replace with your file path

# Drop irrelevant columns and preprocess categorical variables
data_cleaned = data.drop(columns=['time'])  # Remove irrelevant column
data_cleaned = pd.get_dummies(data_cleaned, drop_first=True)  # One-hot encode categorical variables

# Define features (X) and target variable (y)
X = data_cleaned.drop(columns=['power_mW'])  # Features

# Encode target variable for multiclass classification
# Example: Bin power_mW into categories (e.g., low, medium, high)
bins = [0, data_cleaned['power_mW'].quantile(0.33), data_cleaned['power_mW'].quantile(0.66), data_cleaned['power_mW'].max()]
labels = [0, 1, 2]  # Multiclass labels (e.g., low, medium, high)
y = pd.cut(data_cleaned['power_mW'], bins=bins, labels=labels).astype(int)  # Multiclass target

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Initialize and train the XGBoost model
model = xgb.XGBClassifier(objective='multi:softmax', num_class=3, random_state=42)  # Multiclass classification
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model's performance
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')  # Use 'weighted' for multiclass
recall = recall_score(y_test, y_pred, average='weighted')        # Use 'weighted' for multiclass
f1 = f1_score(y_test, y_pred, average='weighted')                # Use 'weighted' for multiclass

# Print evaluation results
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision (weighted): {precision:.2f}")
print(f"Recall (weighted): {recall:.2f}")
print(f"F1-Score (weighted): {f1:.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy: 0.86
Precision (weighted): 0.86
Recall (weighted): 0.86
F1-Score (weighted): 0.85

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.96      0.88     12598
           1       0.81      0.67      0.73     10121
           2       0.93      0.91      0.92     11541

    accuracy                           0.86     34260
   macro avg       0.85      0.84      0.85     34260
weighted avg       0.86      0.86      0.85     34260

